# Обучение и проверка (Colab, GPU)

Тонкий блокнот: весь код в пакете `anemia/` и в `scripts/` репозитория. Запускайте ячейки по порядку.

Архив CP-AnemiC (`CP-AnemiC_dataset.zip`, общая папка проекта `md/history/097_2026-10-02_dataset_cp_anemic/`):
положите в `MyDrive/anemia/` на Диске или перетащите в панель «Файлы» слева; иначе ячейка 2 попросит загрузить его.
Данные и модели хранятся на Диске в `MyDrive/anemia/` (без Диска — в `/content/anemia`), в репозиторий не попадают.

**Обновление кода.** Ячейка 1 при каждом запуске подтягивает `origin/main` (галочка «Обновить_код»), печатает новые коммиты и статус тестов GitHub. Правки, сделанные прямо в `/content/repo`, не теряются: они уходят в `git stash`. Код новых коммитов выполняется только когда вы запускаете ячейки 2–3.

In [ ]:
#@title Ячейка 1 — репозиторий, зависимости, хранилище
#@markdown Google Диск сохраняет данные и модели между сессиями. Если Диск не подключается — снимите галочку:
#@markdown всё будет в /content (стирается при закрытии сессии, данные скачаются заново).
Использовать_Google_Диск = False  #@param {type:"boolean"}
#@markdown **Код из GitHub.** Галочка «Обновить_код» подтягивает свежую версию ветки и печатает список новых коммитов;
#@markdown без галочки остаётся уже скачанная версия. «Коммит» — закрепить конкретный коммит (пусто = последний в ветке).
Обновить_код = True  #@param {type:"boolean"}
Ветка = 'main'  #@param {type:"string"}
Коммит = ''  #@param {type:"string"}
import os, subprocess, sys, time, json, urllib.request
REPO = 'zhansaya-afk/jjghg'
REPO_DIR = '/content/repo'
BRANCH = Ветка.strip() or 'main'
BASE = '/content/anemia'
if Использовать_Google_Диск:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE = '/content/drive/MyDrive/anemia'
    except Exception as e:
        print('⚠️ Google Диск не подключился, работаю без него (данные в /content/anemia):', str(e)[:150])
os.makedirs(BASE, exist_ok=True)
os.environ['ANEMIA_BASE'] = BASE
os.environ['ANEMIA_DATA'] = f'{BASE}/data'
os.environ['ANEMIA_MODELS'] = f'{BASE}/models'

def git(*args):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f'git {" ".join(args)}: {r.stderr.strip()[:300]}')
    return r.stdout.strip()

# токен нужен, только если репозиторий приватный: Colab → ключ слева → Secrets → GITHUB_TOKEN
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], check=True)
    if Коммит.strip():
        git('reset', '-q', '--hard', Коммит.strip())
elif Обновить_код:
    git('fetch', '-q', 'origin', BRANCH)
    target = git('rev-parse', '--verify', (Коммит.strip() or f'origin/{BRANCH}') + '^{commit}')
    stamp = time.strftime('%Y%m%d-%H%M%S')
    # ничего не теряем: правки в /content/repo уходят в stash, свои коммиты — в ветку colab-backup-*
    if git('status', '--porcelain'):
        git('stash', 'push', '-u', '-m', f'colab-{stamp}')
        print(f'⚠️ Локальные правки в {REPO_DIR} сохранены: git -C {REPO_DIR} stash list')
    if git('log', '--oneline', 'HEAD', '--not', target, f'origin/{BRANCH}'):
        git('branch', f'colab-backup-{stamp}')
        print(f'⚠️ Коммиты, которых нет в {BRANCH}, сохранены в ветке colab-backup-{stamp}')
    incoming = git('log', '--oneline', '--no-decorate', f'HEAD..{target}')
    git('reset', '-q', '--hard', target)
    if Коммит.strip():
        print('Закреплён коммит', Коммит.strip())
    print('Новые коммиты:\n' + incoming if incoming else 'Новых коммитов нет.')
else:
    print('Обновление выключено: работаю на уже скачанной версии.')
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data,dev]'], check=True)
sha = git('rev-parse', 'HEAD')
print('Хранилище:', BASE)
print('Коммит:', git('log', '-1', '--format=%h %s (%cd)', '--date=short'))
# статус тестов GitHub Actions для этого коммита (без токена, только для публичного репозитория)
try:
    with urllib.request.urlopen(f'https://api.github.com/repos/{REPO}/commits/{sha}/check-runs', timeout=10) as r:
        runs = json.load(r).get('check_runs', [])
    tests = [c for c in runs if c['name'].startswith('pytest')]
    if not tests:
        print('Тесты GitHub: нет данных для этого коммита')
    elif all(c['conclusion'] == 'success' for c in tests):
        print('Тесты GitHub: ✅ пройдены')
    elif any(c['status'] != 'completed' for c in tests):
        print('Тесты GitHub: ⏳ ещё идут')
    else:
        print('Тесты GitHub: ❌ упали — сообщите Claude, прежде чем запускать дальше')
except Exception as e:
    print('Тесты GitHub: статус не получен:', str(e)[:80])
if 'anemia' in sys.modules:
    print('ℹ️ Пакет anemia уже был импортирован: чтобы взять новый код, «Среда выполнения → Перезапустить сеанс» и снова ячейка 1.')

In [ ]:
#@title Ячейка 2 — данные: скачать (один раз) и собрать manifest.csv
import os, glob, shutil
D, BASE = os.environ['ANEMIA_DATA'], os.environ['ANEMIA_BASE']
if not os.path.exists(f'{D}/eyes_defy_anemia/labels.csv'):
    !python scripts/fetch_data.py --eyes-defy
if not os.path.exists(f'{D}/cp_anemic/labels.csv'):
    # архив CP-AnemiC ищется на Диске (MyDrive/anemia/) или в /content (панель «Файлы» слева);
    # если его нигде нет — появится кнопка загрузки с компьютера
    found = sorted(glob.glob(f'{BASE}/*AnemiC*.zip') + glob.glob('/content/*AnemiC*.zip'))
    if not found:
        try:
            from google.colab import files
            print('Загрузите CP-AnemiC_dataset.zip (5 МБ):')
            up = files.upload()
            found = [os.path.abspath(n) for n in up if n.lower().endswith('.zip')]
        except Exception as e:
            print('⚠️ загрузка не удалась:', str(e)[:120])
    if found:
        cp_zip = found[0]
        !python scripts/fetch_data.py --cp-zip "{cp_zip}"
    else:
        print('⚠️ CP-AnemiC пропущен: нет архива')
!python scripts/prepare_data.py
!python -m pytest -q

In [ ]:
#@title Ячейка 3 — честный baseline (этап 1): сегментация v11 + Hb по цвету, все датасеты
# ≈10–20 мин на CPU/GPU Colab. Пришлите Claude весь вывод ячейки: таблицу и строки CSV в конце.
import os, pandas as pd
before = len(pd.read_csv('results/metrics_history.csv')) if os.path.exists('results/metrics_history.csv') else 0
!python scripts/evaluate.py --tag baseline_v11
h = pd.read_csv('results/metrics_history.csv')
new = h.iloc[before:]
new.to_csv(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv", mode='a', index=False,
           header=not os.path.exists(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv"))
print('\n--- строки для results/metrics_history.csv ---')
print(new.to_csv(index=False))